# Preprocessing Notebook

This notebook prepares analysis-ready tables from the raw YouTube collection outputs.

**How to use:**
- Update `PERSON` below to switch datasets (e.g., `bodeoum_tv` or `mimiminu`).
- Rerun all cells to produce outputs under `02-preprocessing/outputs/<person>/`.


## 0) Imports and configuration
This cell sets the dataset name and all input/output paths.


In [1]:
from __future__ import annotations

import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from tqdm import tqdm

# ---- CONFIG ----
PERSON = "bodeoum_tv"  # change to 'mimiminu' when needed
LANG = "ko"

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent:
    if (REPO_ROOT / "data").exists() and (REPO_ROOT / "02-preprocessing").exists():
        break
    REPO_ROOT = REPO_ROOT.parent
DATA_ROOT = REPO_ROOT / "data"
INPUT_CSV = DATA_ROOT / "input" / f"{PERSON}.csv"
YOUTUBE_JSONL = DATA_ROOT / PERSON / f"{PERSON}_youtube.jsonl"
COMMENTS_DIR = DATA_ROOT / PERSON / "comments"
TRANSCRIPTS_DIR = DATA_ROOT / PERSON / "transcripts"

OUTPUT_DIR = REPO_ROOT / "02-preprocessing" / "outputs" / PERSON
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Using PERSON=", PERSON)
print("Input CSV:", INPUT_CSV)
print("YouTube JSONL:", YOUTUBE_JSONL)
print("Comments dir:", COMMENTS_DIR)
print("Transcripts dir:", TRANSCRIPTS_DIR)
print("Output dir:", OUTPUT_DIR)


Using PERSON= bodeoum_tv
Input CSV: /home/mbchavez/Coding/Research/cancel-culture-sentiment-analysis/data/input/bodeoum_tv.csv
YouTube JSONL: /home/mbchavez/Coding/Research/cancel-culture-sentiment-analysis/data/bodeoum_tv/bodeoum_tv_youtube.jsonl
Comments dir: /home/mbchavez/Coding/Research/cancel-culture-sentiment-analysis/data/bodeoum_tv/comments
Transcripts dir: /home/mbchavez/Coding/Research/cancel-culture-sentiment-analysis/data/bodeoum_tv/transcripts
Output dir: /home/mbchavez/Coding/Research/cancel-culture-sentiment-analysis/02-preprocessing/outputs/bodeoum_tv


## 1) Load window map (CSV)
This builds a `slot -> video_url -> video_id` mapping based on the CSV order.


In [2]:
def extract_video_id(url: str) -> str | None:
    if "watch?v=" in url:
        return url.split("watch?v=", 1)[1].split("&", 1)[0]
    if "youtu.be/" in url:
        return url.split("youtu.be/", 1)[1].split("?", 1)[0]
    return None

window_df = pd.read_csv(INPUT_CSV)
window_df["video_id"] = window_df["video"].astype(str).apply(extract_video_id)

slot_rows = []
for window_label in ("pre", "cancel", "post"):
    rows = window_df[window_df["window"].str.lower() == window_label]
    for i, (_, row) in enumerate(rows.iterrows(), start=1):
        slot = f"{window_label}-{i}"
        slot_rows.append({
            "slot": slot,
            "window": window_label,
            "video": row["video"],
            "video_id": row["video_id"],
        })

slot_df = pd.DataFrame(slot_rows)
slot_df.head()


,slot,window,video,video_id
0,pre-1,pre,https://www.youtube.com/watch?v=eBP5Ds6o2Aw,eBP5Ds6o2Aw
1,pre-2,pre,https://www.youtube.com/watch?v=FCrjawLVIWo,FCrjawLVIWo
2,pre-3,pre,https://www.youtube.com/watch?v=iRo0SSChhwk,iRo0SSChhwk
3,pre-4,pre,https://www.youtube.com/watch?v=L5yvqZfpR64,L5yvqZfpR64
4,pre-5,pre,https://www.youtube.com/watch?v=A1L9uEZ0xNE,A1L9uEZ0xNE


## 2) Extract video metadata (`videos.csv`)
Parses `<person>_youtube.jsonl` and keeps records where `record_type == "video"`.


In [3]:
video_rows = []
with open(YOUTUBE_JSONL, "r", encoding="utf-8") as handle:
    for line in handle:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        if record.get("record_type") != "video":
            continue
        video_rows.append({
            "video_id": record.get("video_id"),
            "title": record.get("title"),
            "video_url": record.get("video_url"),
            "create_time": record.get("create_time"),
            "view_count": record.get("view_count"),
            "like_count": record.get("like_count"),
            "comment_count": record.get("comment_count"),
            "window_label": record.get("window_label"),
        })

videos_df = pd.DataFrame(video_rows)
videos_df.to_csv(OUTPUT_DIR / "videos.csv", index=False)
videos_df.head()


,video_id,title,video_url,create_time,view_count,like_count,comment_count,window_label
0,eBP5Ds6o2Aw,대형견 앞에서 얼어버린 소형견...(대형견=강형욱) [퍼피교육],https://www.youtube.com/watch?v=eBP5Ds6o2Aw,2024-05-18T00:00:00+00:00,1058723,6512,11000,pre
1,FCrjawLVIWo,메롱하면 혓바닥이 보라색인 강아지ㅣ견종백과 차우차우 편,https://www.youtube.com/watch?v=FCrjawLVIWo,2024-05-11T00:00:00+00:00,672091,8599,646,pre
2,iRo0SSChhwk,유기견 출신 친구가 산책하는 법 [집에가자] EP.7,https://www.youtube.com/watch?v=iRo0SSChhwk,2024-05-07T00:00:00+00:00,122673,3263,162,pre
3,L5yvqZfpR64,강형욱에게 트라우마가 된 견종 | 견종백과 와이어 폭스테리어편,https://www.youtube.com/watch?v=L5yvqZfpR64,2024-05-04T00:00:00+00:00,428428,6944,275,pre
4,A1L9uEZ0xNE,"엄정화님, 도대체 어떻게 키우신 건가요. 강형욱이 사랑에 빠진 진돗개 등장[강형욱의...",https://www.youtube.com/watch?v=A1L9uEZ0xNE,2024-04-27T00:00:00+00:00,2118018,37397,1900,pre


## 3) Flatten comments (`comments.csv`)
Reads `comments/*.info.json` and extracts comment fields + window slot.


In [4]:
comment_rows = []
comment_files = sorted(COMMENTS_DIR.glob("*.info.json"))

for path in tqdm(comment_files, desc="comments"):
    slot = path.stem.replace(".info", "")
    window = slot.split("-")[0]
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)

    video_id = payload.get("id")
    comments = payload.get("comments", [])
    for c in comments:
        comment_rows.append({
            "comment_id": c.get("id"),
            "parent_id": c.get("parent"),
            "author": c.get("author"),
            "author_id": c.get("author_id"),
            "timestamp": c.get("timestamp"),
            "like_count": c.get("like_count"),
            "text": c.get("text"),
            "video_id": video_id,
            "window": window,
            "slot": slot,
        })

comments_df = pd.DataFrame(comment_rows)
comments_df.to_csv(OUTPUT_DIR / "comments.csv", index=False)
comments_df.head()


comments:   0%|                                                  | 0/21 [00:00<?, ?it/s]

comments:   5%|██                                        | 1/21 [00:01<00:32,  1.65s/it]

comments:  14%|██████                                    | 3/21 [00:01<00:08,  2.15it/s]

comments:  38%|████████████████                          | 8/21 [00:01<00:01,  7.01it/s]

comments:  57%|███████████████████████▍                 | 12/21 [00:02<00:00, 10.18it/s]

comments:  81%|█████████████████████████████████▏       | 17/21 [00:02<00:00, 14.94it/s]

comments: 100%|█████████████████████████████████████████| 21/21 [00:02<00:00,  9.33it/s]

,comment_id,parent_id,author,author_id,timestamp,like_count,text,video_id,window,slot
0,UgyOcO5J0AjmoZcSxzF4AaABAg,root,@young-qj5kk,UCT03lvLLDVH-svq9Nxij4gQ,1776603600,0,자기네 할 일만하고 성실하게 한다면 CCTV를 신경쓸일이 아닐텐데ㅠㅠ,e5N9eXLcfag,cancel,cancel-1
1,UgxS4Vbyo9X--Gq2RpV4AaABAg,root,@abcdefg-j8g7r,UC1z4zEb3aV0I5eMjL6EB3xg,1776556800,1,회사에서 일 하고 회사 앱 쓰면 조심 해야지!! 그런 내용을 봤던 안 봤던 왜 회사...,e5N9eXLcfag,cancel,cancel-1
2,Ugy65VLiNwfhVdpTbh54AaABAg,root,@구르맨,UCoIprje599dMhf4F9QV3oPw,1776470400,0,중견 대기업도 스트레스인데 중소회사에서 뭘 바라는게 많냐,e5N9eXLcfag,cancel,cancel-1
3,UgzVLXl_q81dBdlOG9p4AaABAg,root,@ElenaLee-r2f,UCpObO9VgV5YhPRJWouc0ncw,1776470400,1,요새 강형욱님 영상 많이 떠서 보다가 여기까지 왔네요.. 일년전 영상인데...힘든 ...,e5N9eXLcfag,cancel,cancel-1
4,Ugxml-oKBzfTE7IMHlB4AaABAg,root,@motivateJmusic,UCNRSxiMCaGo0AYwdcavvkSQ,1776384000,1,가끔보면 느껴진다....인간도 훈련이 필요한 사람들이 있다는걸....정말 악한 인간...,e5N9eXLcfag,cancel,cancel-1


## 4) Build reply edges (`edges.csv`)
Creates reply graph edges from `comments.csv`.


In [5]:
edges_df = comments_df[comments_df["parent_id"].notna() & (comments_df["parent_id"] != "root")].copy()
edges_df = edges_df.rename(columns={
    "comment_id": "source_comment_id",
    "parent_id": "target_comment_id",
})
edges_df = edges_df[[
    "source_comment_id",
    "target_comment_id",
    "author_id",
    "video_id",
    "window",
    "slot",
]]
edges_df.to_csv(OUTPUT_DIR / "edges.csv", index=False)
edges_df.head()


,source_comment_id,target_comment_id,author_id,video_id,window,slot
38,UgxVZWuvvmi4iVl2jAx4AaABAg.ATrAmc99c4XATv0hhYAkdP,UgxVZWuvvmi4iVl2jAx4AaABAg,UCpYT0zityNsC8GK0syABhdg,e5N9eXLcfag,cancel,cancel-1
39,UgxVZWuvvmi4iVl2jAx4AaABAg.ATrAmc99c4XAUM4DBIjy3x,UgxVZWuvvmi4iVl2jAx4AaABAg,UCIwEwBK5rhHo4AKtcjr9Yng,e5N9eXLcfag,cancel,cancel-1
59,UgzYw7vZFIqPIjrbvnB4AaABAg.ASRn5GVu0XpASUfIosOcFh,UgzYw7vZFIqPIjrbvnB4AaABAg,UCAz_AfPpz-QnhxsnWI_CPSQ,e5N9eXLcfag,cancel,cancel-1
60,UgzYw7vZFIqPIjrbvnB4AaABAg.ASRn5GVu0XpASWsZAeuMON,UgzYw7vZFIqPIjrbvnB4AaABAg.ASRn5GVu0XpASUfIosOcFh,UCepcRSj5K8YGpF_W3iC1WAQ,e5N9eXLcfag,cancel,cancel-1
64,UgxRmeVPONaUnH3cGH54AaABAg.AS-RNkYGR4HAS90uFzT1Vw,UgxRmeVPONaUnH3cGH54AaABAg,UCB0lojyk0T0SuR3qk9p-NsA,e5N9eXLcfag,cancel,cancel-1


## 5) Parse transcripts (`transcripts.csv`)
Reads `transcripts/*.ko.txt` and attaches window/slot.


In [6]:
transcript_rows = []
for path in sorted(TRANSCRIPTS_DIR.glob(f"*.{LANG}.txt")):
    name = path.name
    # expected: pre-1.ko.txt / cancel-1.ko.txt / post-1.ko.txt
    slot = name.split(".")[0]
    window = slot.split("-")[0]
    text = path.read_text(encoding="utf-8")
    transcript_rows.append({
        "slot": slot,
        "window": window,
        "transcript_text": text,
    })

transcripts_df = pd.DataFrame(transcript_rows)
transcripts_df.to_csv(OUTPUT_DIR / "transcripts.csv", index=False)
transcripts_df.head()


,slot,window,transcript_text
0,cancel-1,cancel,안녕하세요. 훈련사 강영웅입니다.\n사실 여부를 따지기에 앞서서\n이런 소식으로 좀...


## 6) Text cleaning
Creates `text_clean` for comments and transcripts without overwriting raw text.


In [7]:
import re

url_re = re.compile(r"https?://\S+")
whitespace_re = re.compile(r"\s+")

def clean_text(text: str | None) -> str:
    if not text:
        return ""
    text = url_re.sub("", text)
    text = whitespace_re.sub(" ", text).strip()
    return text

comments_df["text_clean"] = comments_df["text"].astype(str).apply(clean_text)
transcripts_df["text_clean"] = transcripts_df["transcript_text"].astype(str).apply(clean_text)

comments_df.to_csv(OUTPUT_DIR / "comments.csv", index=False)
transcripts_df.to_csv(OUTPUT_DIR / "transcripts.csv", index=False)

comments_df[["text", "text_clean"]].head()


,text,text_clean
0,자기네 할 일만하고 성실하게 한다면 CCTV를 신경쓸일이 아닐텐데ㅠㅠ,자기네 할 일만하고 성실하게 한다면 CCTV를 신경쓸일이 아닐텐데ㅠㅠ
1,회사에서 일 하고 회사 앱 쓰면 조심 해야지!! 그런 내용을 봤던 안 봤던 왜 회사...,회사에서 일 하고 회사 앱 쓰면 조심 해야지!! 그런 내용을 봤던 안 봤던 왜 회사...
2,중견 대기업도 스트레스인데 중소회사에서 뭘 바라는게 많냐,중견 대기업도 스트레스인데 중소회사에서 뭘 바라는게 많냐
3,요새 강형욱님 영상 많이 떠서 보다가 여기까지 왔네요.. 일년전 영상인데...힘든 ...,요새 강형욱님 영상 많이 떠서 보다가 여기까지 왔네요.. 일년전 영상인데...힘든 ...
4,가끔보면 느껴진다....인간도 훈련이 필요한 사람들이 있다는걸....정말 악한 인간...,가끔보면 느껴진다....인간도 훈련이 필요한 사람들이 있다는걸....정말 악한 인간...


## 7) Summary
Quick sanity checks on row counts and output paths.


In [8]:
print("videos.csv", len(videos_df))
print("comments.csv", len(comments_df))
print("edges.csv", len(edges_df))
print("transcripts.csv", len(transcripts_df))
print("Output dir:", OUTPUT_DIR)


videos.csv 85
comments.csv 129406
edges.csv 42738
transcripts.csv 1
Output dir: /home/mbchavez/Coding/Research/cancel-culture-sentiment-analysis/02-preprocessing/outputs/bodeoum_tv
